### Cell 1: Imports & Environment Setup

*Install required libraries if you haven't already*

In [1]:
!pip install scikit-learn sentence-transformers pandas numpy

In [2]:
# Local directories to hold models and uploads
import os
from pathlib import Path

HOME = Path.home()

MODELS_DIR = HOME / "ai_models"
UPLOADS_DIR = HOME / "ai_uploads"
LOGS_DIR = "../logs"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(UPLOADS_DIR, exist_ok=True)

In [3]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer

# Verify file path
log_path = os.path.join(LOGS_DIR, "train_auth.log")
print(f"Checking for log file at: {os.path.abspath(log_path)}")
if os.path.exists(log_path):
    print("Log file found successfully!")
else:
    print("WARNING: Log file not found. Make sure ../logs/auth.log exists.")

Checking for log file at: /home/jovyan/AI_examples/logs/train_auth.log
Log file found successfully!


In [4]:
# Check for GPU (cuda) or CPU
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            1
        ),
        "GB"
    )

CUDA: False


### Cell 2: Load and Parse the Log File

This cell reads the raw log file and uses regex to strip dynamic PIDs, specific usernames, and timestamps so the model learns the **structural template** of the log rather than memorizing individual machine noise.

In [5]:
def normalize_log_line(line):
    # Strip timestamp (e.g., 2026-11-12T08:43:34.993955-08:00)
    line = re.sub(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+-\d{2}:\d{2}\s+', '', line)
    # Strip hostname (e.g., xps )
    line = re.sub(r'^\S+\s+', '', line)
    # Normalize PIDs (e.g., [1349051] -> [PID])
    line = re.sub(r'\[\d+\]', '[PID]', line)
    # Normalize UIDs and specific users where helpful
    line = re.sub(r'uid=\d+', 'uid=ID', line)
    return line.strip()

raw_lines = []
normalized_lines = []

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        for line in f:
            if line.strip():
                raw_lines.append(line.strip())
                normalized_lines.append(normalize_log_line(line))

df = pd.DataFrame({
    'raw_log': raw_lines,
    'normalized_log': normalized_lines
})

print(f"Loaded {len(df)} log lines.")
df.head()

Loaded 507 log lines.


,raw_log,normalized_log
0,2026-11-01T12:40:18.762049-08:00 xps sudo: pam...,sudo: pam_unix(sudo:session): session opened f...
1,2026-11-01T12:40:18.762117-08:00 xps sudo: roo...,sudo: root : PWD=/ ; USER=root ; COMMAND=/usr...
2,2026-11-01T12:40:19.374100-08:00 xps sudo: pam...,sudo: pam_unix(sudo:session): session closed f...
3,2026-11-01T12:40:19.386198-08:00 xps sudo: pam...,sudo: pam_unix(sudo:session): session opened f...
4,2026-11-01T12:40:19.386255-08:00 xps sudo: roo...,sudo: root : PWD=/ ; USER=root ; COMMAND=/usr...


### Cell 3: Feature Engineering & Category Labeling (Lab Stage 1)

To train a category classifier, we label the logs based on their source daemon/service (`CRON`, `gdm-password`, `systemd-machined`, etc.).

In [6]:
def assign_category(log):
    log_lower = log.lower()
    
    # 1. Security events (sudo escalations, failures, anomalies)
    if any(keyword in log_lower for keyword in ["sudo", "fail", "invalid", "auth", "error", "denied"]):
        return "Security"
    
    # 2. Authentication events (logins, keyrings, pam sessions)
    elif any(keyword in log_lower for keyword in ["gdm-password", "keyring", "pam_unix", "password", "session opened", "session closed"]):
        return "Authentication"
    
    # 3. Service lifecycle events (cron jobs, daemons)
    elif any(keyword in log_lower for keyword in ["cron", "systemd", "started", "stopped"]):
        return "Service"
    
    # 4. Configuration or environment changes
    elif any(keyword in log_lower for keyword in ["config", "machine", "new machine"]):
        return "Configuration"
        
    # 5. Default fallback for standard text lines
    else:
        return "Plain Text"

# Apply the updated function to your dataframe
df['category'] = df['normalized_log'].apply(assign_category)

print("Updated Category Distribution:")
print(df['category'].value_counts())

Updated Category Distribution:
category
Authentication    364
Security           82
Service            53
Plain Text          8
Name: count, dtype: int64


### Cell 4: Generate Embeddings & Train the Categorizer

Using a small, high-performance embedding model (`all-MiniLM-L6-v2`), we convert log templates into dense vectors, then train a fast Scikit-Learn logistic regression classifier.

In [7]:
# Load a fast, lightweight sentence transformer model
print("Loading embedding model...")
model = SentenceTransformer(
    'all-MiniLM-L6-v2',
    cache_folder = MODELS_DIR,
    device = DEVICE
)

# Generate vector embeddings for the normalized logs
print("Generating embeddings...")
X = model.encode(df['normalized_log'].tolist(), show_progress_bar=True)
y = df['category'].to_numpy()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [8]:
# Split into train and test sets (if we have enough diversity)
if len(df) > 5:
    # Split the DataFrame directly so we can keep the text strings attached to test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    classifier = LogisticRegression(max_iter=1000)
    # Train the classifier on train data
    classifier.fit(X_train, y_train)

    # Predict on the hidden test set
    y_pred = classifier.predict(X_test)
    print("\nModel Evaluation Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
else:
    print("Dataset too small for train/test split. Training on all available data.")
    classifier = LogisticRegression(max_iter=1000)
    classifier.fit(X, y)


Model Evaluation Report:
                precision    recall  f1-score   support

Authentication       0.99      1.00      0.99        71
    Plain Text       1.00      1.00      1.00         3
      Security       1.00      0.91      0.95        11
       Service       1.00      1.00      1.00        17

      accuracy                           0.99       102
     macro avg       1.00      0.98      0.99       102
  weighted avg       0.99      0.99      0.99       102



In [9]:
# # 6. Print line-by-line inspection results
# print("\n=== Line-by-Line Test Inspection ===")
# print(f"{'RESULT':<8} | {'ACTUAL CATEGORY':<16} | {'PREDICTED':<16} | {'NORMALIZED LOG LINE'}")
# print("-" * 90)

# for actual, pred, log_line in zip(y_test, y_pred, df_test['normalized_log']):
#     match_status = "✅ PASS" if actual == pred else "❌ FAIL"
#     # Truncate long log lines if needed for clean printing
#     short_log = log_line if len(log_line) < 45 else log_line[:42] + "..."
#     print(f"{match_status:<8} | {actual:<16} | {pred:<16} | {short_log}")

### Cell 5: Stage 2 - Simulating Malicious Threat Detection

To transition your lab into identifying **malicious vs. benign** logs, we can simulate an injection of malicious auth events (such as SSH brute-force failures or unauthorized `sudo` attempts) and train a binary classifier.

In [14]:
# Create synthetic training data combining your real benign logs and simulated malicious threats
benign_samples = df['normalized_log'].tolist()

malicious_samples = [
    # --- SSH Brute-Force & Invalid User Attacks ---
    "sshd[PID]: Failed password for invalid user admin from 192.168.1.100 port 51234 ssh2",
    "sshd[PID]: Failed password for root from 10.0.0.5 port 43210 ssh2",
    "sshd[PID]: Failed password for invalid user test from 203.0.113.50 port 39821 ssh2",
    "sshd[PID]: Failed password for invalid user oracle from 172.16.25.10 port 44122 ssh2",
    "sshd[PID]: Invalid user support from 45.33.32.156 port 58210",
    "sshd[PID]: Connection closed by authenticating user root 192.168.1.100 port 51234 [preauth]",
    
    # --- Sudo & Privilege Escalation Failures ---
    "sudo: pam_unix(sudo:auth): authentication failure; logname=nobody uid=1000 euid=0 tTY=/dev/pts/1 ruser= rhost=user-pc",
    "sudo: 3 incorrect password attempts; USER=root ; COMMAND=/bin/bash",
    "sudo: account www-data is not allowed to execute sudo commands; TTY=unknown ; PWD=/var/www ; USER=root ; COMMAND=/bin/su",
    "su: pam_authenticate: authentication failure; logname=www-data uid=33 euid=0 tty=/dev/pts/2 ruser=www-data rhost=",
    
    # --- Scanning & Protocol Anomalies ---
    "sshd[PID]: Did not receive identification string from 192.168.1.200 port 34567",
    "sshd[PID]: Read error from remote host 10.0.0.99 port 22: Connection reset by peer [preauth]",
    "sshd[PID]: Received disconnect from 198.51.100.25 port 60123:1: Bye Bye [preauth]",
    "sshd[PID]: PAM 2 more authentication failures; logname= uid=0 euid=0 tty=ssh rhost=185.199.108.153  user=root"
]

# Combine datasets and create binary labels (0 = Benign, 1 = Malicious)
all_logs = benign_samples + malicious_samples
labels = [0] * len(benign_samples) + [1] * len(malicious_samples)

# Embed all logs
print("Encoding dataset for threat detection...")
X_threat = model.encode(all_logs)
y_threat = np.array(labels)

# Train a binary threat classifier
X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(X_threat, y_threat, test_size=0.2, random_state=42, stratify=y_threat)

threat_classifier = LogisticRegression(max_iter=1000, class_weight='balanced')
threat_classifier.fit(X_t_train, y_t_train)

y_t_pred = threat_classifier.predict(X_t_test)
print("\nThreat Classifier Performance:")
print(classification_report(y_t_test, y_t_pred))

Encoding dataset for threat detection...

Threat Classifier Performance:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       102
           1       1.00      1.00      1.00         3

    accuracy                           1.00       105
   macro avg       1.00      1.00      1.00       105
weighted avg       1.00      1.00      1.00       105



### Cell 6: Test a Live Log Stream Inference

Test your trained models against new or unseen log lines to show students how fast and accurate purpose-built models are.

In [15]:
test_logs = [
    "2026-11-12T10:45:00.123456-08:00 xps CRON[999999]: pam_unix(cron:session): session opened for user root",
    "2026-11-12T11:00:00.654321-08:00 xps sshd[8888]: Failed password for root from 203.0.113.50 port 22 ssh2"
]

for log in test_logs:
    norm = normalize_log_line(log)
    embedding = model.encode([norm])
    
    category = classifier.predict(embedding)[0]
    threat_pred = threat_classifier.predict(embedding)[0]
    threat_label = "🚨 MALICIOUS / ANOMALY" if threat_pred == 1 else "✅ BENIGN"
    
    print(f"Log: {log}")
    print(f"  -> Category: {category}")
    print(f"  -> Status: {threat_label}\n")

Log: 2026-11-12T10:45:00.123456-08:00 xps CRON[999999]: pam_unix(cron:session): session opened for user root
  -> Category: Authentication
  -> Status: ✅ BENIGN

Log: 2026-11-12T11:00:00.654321-08:00 xps sshd[8888]: Failed password for root from 203.0.113.50 port 22 ssh2
  -> Category: Authentication
  -> Status: 🚨 MALICIOUS / ANOMALY



## Classify additional logs

Once model is trained, classify another log and report

In [16]:
from collections import Counter

def analyze_security_log_file(file_path, category_classifier, threat_classifier, embedding_model, normalize_func):
    """
    Reads a new log file, normalizes lines, predicts their category and threat status,
    and returns a full security summary report.
    """
    if not os.path.exists(file_path):
        print(f"❌ Error: File not found at {file_path}")
        return None
    
    raw_lines = []
    normalized_lines = []
    
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.strip():
                raw_lines.append(line.strip())
                normalized_lines.append(normalize_func(line))
                
    if not normalized_lines:
        print("The log file is empty.")
        return None

    print(f"📖 Read {len(normalized_lines)} lines from {file_path}")
    
    # 1. Generate embeddings using the shared embedding model
    print("Generating embeddings for log analysis...")
    X_logs = embedding_model.encode(normalized_lines, show_progress_bar=False)
    
    # 2. Predict both Category and Threat Status (0 = Benign, 1 = Malicious)
    categories = category_classifier.predict(X_logs)
    threat_preds = threat_classifier.predict(X_logs)
    
    # Map binary threat predictions to readable labels
    threat_labels = ["✅ Benign" if p == 0 else "🚨 Malicious/Anomaly" for p in threat_preds]
    
    # 3. Aggregate statistics
    total_logs = len(raw_lines)
    category_counts = Counter(categories)
    threat_counts = Counter(threat_labels)
    
    # 4. Print Summary Report
    print("\n" + "="*50)
    print(f"🛡️ SECURITY SUMMARY REPORT: {os.path.basename(file_path)}")
    print("="*50)
    print(f"Total Log Lines Scanned: {total_logs}")
    
    print("\n--- Threat Assessment Breakdown ---")
    for status, count in threat_counts.items():
        percentage = (count / total_logs) * 100
        print(f"  • {status}: {count} lines ({percentage:.1f}%)")
        
    print("\n--- Category Breakdown ---")
    for cat, count in category_counts.most_common():
        percentage = (count / total_logs) * 100
        print(f"  • {cat}: {count} lines ({percentage:.1f}%)")
    print("="*50)
    
    # Return detailed DataFrame for inspecting flagged malicious lines easily
    results_df = pd.DataFrame({
        'raw_log': raw_lines,
        'normalized_log': normalized_lines,
        'category': categories,
        'threat_status': threat_labels
    })
    return results_df

# --- RUN IT ---
# Pass both your 'classifier' and 'threat_classifier' into the function
security_report_df = analyze_security_log_file("../logs/auth.log.2", classifier, threat_classifier, model, normalize_log_line)

📖 Read 157 lines from ../logs/auth.log.2
Generating embeddings for log analysis...

🛡️ SECURITY SUMMARY REPORT: auth.log.2
Total Log Lines Scanned: 157

--- Threat Assessment Breakdown ---
  • ✅ Benign: 157 lines (100.0%)

--- Category Breakdown ---
  • Authentication: 120 lines (76.4%)
  • Security: 21 lines (13.4%)
  • Service: 13 lines (8.3%)
  • Plain Text: 3 lines (1.9%)


In [17]:
# Filter and view only malicious or anomalous lines detected in the file
malicious_findings = security_report_df[security_report_df['threat_status'] == "🚨 Malicious/Anomaly"]

print(f"🚨 Found {len(malicious_findings)} suspicious/malicious log lines:")
if len(malicious_findings) > 0:
    for idx, row in malicious_findings.iterrows():
        print(f"[{row['category']}] {row['raw_log']}")
else:
    print("No malicious lines detected in this file.")

🚨 Found 0 suspicious/malicious log lines:
No malicious lines detected in this file.
